# ViFinQA — BGE-M3 rows-first held-out ablation V1

Research-only evaluation of `passage_layout=rows_first` at `max_seq_length=384` on synthetic issuer-held-out validation/test rows. The notebook clones one immutable Git tag through a Kaggle Secret, verifies the source tree, and never promotes a model.

In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys
os.environ['WANDB_DISABLED']='true'; os.environ['WANDB_MODE']='disabled'
REPO_DIR=Path('/kaggle/working/AI_guru_rows_first_ablation')
EXPECTED_COMMIT='bbdbf23'
SOURCE_REF='synthetic-retriever-rows-first-source-v1'
if REPO_DIR.exists(): raise RuntimeError(f'Refusing to overwrite {REPO_DIR}')
try:
    from kaggle_secrets import UserSecretsClient
    secrets=UserSecretsClient(); token=None
    for name in ('GITHUB_TOKEN','GIT_TOKEN'):
        try: token=secrets.get_secret(name)
        except Exception: continue
        if token: break
except Exception as exc:
    raise RuntimeError('Kaggle Secret GITHUB_TOKEN (or GIT_TOKEN) is required.') from exc
if not token: raise RuntimeError('Missing Kaggle Secret GITHUB_TOKEN (or GIT_TOKEN).')
env=os.environ.copy(); env['GIT_TERMINAL_PROMPT']='0'; env['GIT_CONFIG_NOSYSTEM']='1'; env['GITHUB_TOKEN']=token
subprocess.run(['git','clone','--depth','1','--branch',SOURCE_REF,'https://github.com/Dle28/nlp-finance-query-.git',str(REPO_DIR)],env=env,check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE,text=True)
revision=subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','HEAD'],text=True).strip()
if not revision.startswith(EXPECTED_COMMIT): raise ValueError(f'Expected commit {EXPECTED_COMMIT}, got {revision}')
files={p.relative_to(REPO_DIR).as_posix():hashlib.sha256(p.read_bytes()).hexdigest() for p in REPO_DIR.rglob('*') if p.is_file() and '.git' not in p.parts and p.relative_to(REPO_DIR).as_posix()!='pax_global_header'}
if any(p=='data/ViFinQA' or p.startswith('data/ViFinQA/') for p in files): raise ValueError('Pinned source contains benchmark data.')
tree_sha=hashlib.sha256(json.dumps(files,sort_keys=True,separators=(',',':')).encode()).hexdigest()
evaluator=REPO_DIR/'scripts/evaluate_synthetic_retriever_v1.py'
if not evaluator.is_file(): raise FileNotFoundError('Pinned source lacks evaluator.')
print({'commit':revision,'source_tree_sha256':tree_sha,'files':len(files)})

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','torch==2.12.1','torchvision==0.27.1','--index-url','https://download.pytorch.org/whl/cu126'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','sentence-transformers==3.4.1','transformers==4.48.3'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO_DIR),'--no-deps'],check=True)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Enable Kaggle GPU and restart.')
vram_gib=torch.cuda.get_device_properties(0).total_memory/1024**3
if vram_gib<14: raise RuntimeError(f'Requires >=14 GiB VRAM; found {vram_gib:.1f} GiB')
print({'gpu':torch.cuda.get_device_name(0),'vram_gib':round(vram_gib,2),'torch':torch.__version__})

In [ ]:
INPUT_ROOT=Path('/kaggle/input')
CURRICULUM_NAME='synthetic_finance_curriculum_v1.jsonl'; MANIFEST_NAME='synthetic_finance_curriculum_v1.manifest.json'
curriculum_dirs=sorted({p.parent for p in INPUT_ROOT.rglob(CURRICULUM_NAME) if (p.parent/MANIFEST_NAME).is_file()})
if len(curriculum_dirs)!=1: raise RuntimeError(f'Expected one curriculum input, found {len(curriculum_dirs)}')
CURRICULUM_DIR=curriculum_dirs[0]; CURRICULUM=CURRICULUM_DIR/CURRICULUM_NAME; CURRICULUM_MANIFEST=CURRICULUM_DIR/MANIFEST_NAME
tables=sorted(INPUT_ROOT.rglob('table_assets.jsonl'))
if len(tables)!=1: raise RuntimeError(f'Expected one table_assets.jsonl, found {len(tables)}')
TABLES=tables[0]
model_dirs=sorted({p.parent for p in INPUT_ROOT.rglob('training_metadata.json') if (p.parent/'model.safetensors').is_file() and (p.parent/'modules.json').is_file()})
if len(model_dirs)!=1: raise RuntimeError(f'Expected one trained model output, found {len(model_dirs)}')
FINETUNED_MODEL=model_dirs[0]; training_metadata=json.loads((FINETUNED_MODEL/'training_metadata.json').read_text())
if training_metadata.get('provenance')!='synthetic_execution_verified': raise ValueError('Unexpected trained-model provenance')
print({'curriculum':str(CURRICULUM),'tables':str(TABLES),'finetuned_model':str(FINETUNED_MODEL)})

## Controlled ablation

Only passage order changes. The evaluator remains issuer-held-out, hash-bound, and non-promotable.

In [ ]:
OUTPUT_DIR=Path('/kaggle/working/bge_m3_rows_first_heldout_ablation_v1')
command=[sys.executable,str(evaluator),'--curriculum',str(CURRICULUM),'--manifest',str(CURRICULUM_MANIFEST),'--bundle-tables',str(TABLES),'--output-dir',str(OUTPUT_DIR),'--model','base=BAAI/bge-m3','--model',f'finetuned={FINETUNED_MODEL}','--splits','validation','test','--ks','1','3','5','10','20','--passage-batch-size','16','--query-batch-size','32','--max-seq-length','384','--passage-layout','rows_first','--device','cuda:0']
print('Running:', ' '.join(command)); subprocess.run(command,cwd=REPO_DIR,check=True)

In [ ]:
result=json.loads((OUTPUT_DIR/'evaluation_manifest.json').read_text())
if result.get('promotion_status')!='offline_evaluation_complete_not_promoted': raise ValueError('Unexpected promotion status')
if result.get('configuration',{}).get('passage_layout')!='rows_first': raise ValueError('Rows-first layout not recorded')
summary={label:{split:{'mrr':round(m['mrr'],4),'recall@10':round(m['recall_at_k']['10'],4),'top1_errors':m['top1_error_counts']} for split,m in model_result['splits'].items()} for label,model_result in result['models'].items()}
print(json.dumps({'summary':summary,'status':result['promotion_status'],'artifacts':sorted(p.name for p in OUTPUT_DIR.iterdir())},ensure_ascii=False,indent=2))

## Decision boundary

Use this artifact only to compare against the prior context-first baseline. Accept rows-first for later training only if both held-out splits improve and wrong-year/wrong-scope rates do not worsen.